In [34]:
import pandas as pd
import numpy as np
import glob

from collections import defaultdict
from typing import List
from metrics import assert_valid_prob, assert_same_exprs, compute_wasserstein_distance, compute_mean_conf_error, compute_proportional_agreement
from utils_io import read_json
from utils_latex import *
from default_vars import UNCERTAINTY_EXPRESSIONS

EXCLUDE_MODELS = [
    "full__lmsys__vicuna-13b-v1.5",
    "full__mistralai__Mistral-7B-Instruct-v0.2",
    "sampling__allenai__OLMo-7B-Instruct","sampling__google__gemma-1.1-2b-it"
]

In this notebook, we will compute the two sets of metrics: 
- _mode-matching_ metrics, which include the `proportional agreement` and `mean absolute error`.
- _distribution matching_ metrics: which include the `wasserstein` distance.

## 1. Wasserstein Distance

Since we already stored the normalized histograms, it is relatively trivial to compute the wasserstein distance across different histograms. The final reported result is the median of the Wasserstein distance for all 14 uncertainty expressions.

In [35]:
human_ref_nv = pd.read_csv("../../results/greedy/all/non_verifiable/humans-2shot-filtered/normalized.csv", index_col=0)
# Check that normalized histograms sum to approx 1
assert_valid_prob(human_ref_nv)

human_ref_v = pd.read_csv("../../results/greedy/all/verifiable/humans-2shot-filtered/normalized.csv", index_col=0)
# Check that normalized histograms sum to approx 1
assert_valid_prob(human_ref_v)

# Check that both files concern the same uncertainty expressions, ordered in the same way
assert_same_exprs(human_ref_v, human_ref_nv)
compute_wasserstein_distance(human_ref_nv, human_ref_v, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)

,uncertainty_expression,distance
0,almost certain,1.907722
1,highly likely,1.351901
2,very likely,3.441908
3,probable,1.316639
4,somewhat likely,2.234939
5,possible,2.324886
6,uncertain,2.751913
7,somewhat unlikely,2.009622
8,unlikely,2.088812
9,not likely,1.778628


In [36]:
_non_verifiable_results_ws = []
for n_shots in (0, 2):
    model_filepaths = sorted(glob.glob(f"../../results/greedy/all/non_verifiable/models-{n_shots}shot/*_normalized.csv"))  
    for fp in model_filepaths:
        print("Processing", fp)
        human_df = pd.read_csv("../../results/greedy/all/non_verifiable/humans-2shot-filtered/normalized.csv", index_col=0)
        model_df = pd.read_csv(fp, index_col=0)
        dist = compute_wasserstein_distance(model_df, human_df, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)
        dist["n_shots"] = n_shots
        dist["setting"] = "non-verifiable"
        
        model_name = fp.rpartition("shot/")[-1].rpartition("_normalized")[0]
        dist["model"] = model_name
        _non_verifiable_results_ws.append(dist)

_non_verifiable_results_ws = pd.concat(_non_verifiable_results_ws, axis=0).reset_index(drop=True)

Processing ../../results/greedy/all/non_verifiable/models-0shot/full__allenai__OLMo-7B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__google__gemma-1.1-2b-it_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__lmsys__vicuna-13b-v1.5_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__meta-llama__Meta-Llama-3-70B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__meta-llama__Meta-Llama-3-8B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__mistralai__Mistral-7B-Instruct-v0.2_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-4o-2024-05-13_normalized.csv
Processin

In [37]:
print("2 shot")
_non_verifiable_results_ws[_non_verifiable_results_ws["n_shots"]==2].pivot(index="uncertainty_expression", columns="model", values="distance")

2 shot


model,full__allenai__OLMo-7B-Instruct,full__google__gemma-1.1-2b-it,full__lmsys__vicuna-13b-v1.5,full__meta-llama__Meta-Llama-3-70B-Instruct,full__meta-llama__Meta-Llama-3-8B-Instruct,full__mistralai__Mistral-7B-Instruct-v0.2,gpt-3.5-turbo-0125,gpt-4-turbo-2024-04-09,gpt-4o-2024-05-13,sampling__meta-llama__Llama-3-70b-chat-hf,sampling__mistralai__Mixtral-8x22B-Instruct-v0.1,sampling__mistralai__Mixtral-8x7B-Instruct-v0.1
uncertainty_expression,,,,,,,,,,,,
almost certain,3.053191,36.512411,7.884752,3.906028,3.989362,1.943262,3.156028,3.989362,3.418440,3.739362,3.989362,3.906028
doubtful,35.003546,14.734043,18.375887,9.102837,21.829787,24.874113,4.771277,11.459220,9.547872,8.925532,9.769504,10.870567
highly likely,5.028369,28.689716,11.317376,8.253546,8.257092,20.416667,7.523050,7.336879,8.586879,11.757092,8.297872,4.253546
highly unlikely,31.079787,25.349291,12.634752,10.129433,17.496454,13.551418,6.884752,10.212766,9.698582,10.212766,10.904255,9.070922
not likely,28.574468,16.765957,18.992908,8.652482,10.093972,15.365248,6.379433,11.735816,10.489362,10.485816,9.046099,7.921986
possible,10.803191,17.335106,20.218085,16.886525,26.117021,16.952128,12.420213,17.553191,24.533688,16.650709,27.281915,15.386525
probable,6.129433,22.239362,24.303191,8.230496,16.281915,23.799645,14.808511,10.237589,9.070922,7.340426,10.397163,10.097518
somewhat likely,5.624113,29.168440,42.689716,11.303191,16.856383,18.356383,15.012411,9.404255,11.241135,11.303191,9.487589,6.374113
somewhat unlikely,34.250000,7.280142,9.870567,9.494681,8.328014,20.469858,9.078014,7.815603,8.241135,9.494681,7.526596,6.815603


In [38]:
ws_dist_avg = _non_verifiable_results_ws[
    ~(_non_verifiable_results_ws["model"].isin(EXCLUDE_MODELS))
].drop(["setting", "uncertainty_expression"], axis=1)\
 .groupby(["n_shots", "model"])\
 .mean().reset_index().pivot(index="model", columns="n_shots", values="distance")
ws_dist_avg

n_shots,0,2
model,,
full__allenai__OLMo-7B-Instruct,17.850518,21.682351
full__google__gemma-1.1-2b-it,29.025914,21.737589
full__meta-llama__Meta-Llama-3-70B-Instruct,11.372204,9.616612
full__meta-llama__Meta-Llama-3-8B-Instruct,11.051009,13.842744
gpt-3.5-turbo-0125,10.078696,8.895526
gpt-4-turbo-2024-04-09,9.352837,10.034370
gpt-4o-2024-05-13,9.622613,10.457038
sampling__meta-llama__Llama-3-70b-chat-hf,NaN,10.101200
sampling__mistralai__Mixtral-8x22B-Instruct-v0.1,11.401118,11.000000


## 2. Proportional Agreement

In [39]:
_non_verifiable_results_pa = []

for n_shots in (0, 2):
    model_filepaths = sorted(glob.glob(f"../../results/greedy/all/non_verifiable/models-{n_shots}shot/*_normalized.csv"))  
    for fp in model_filepaths:
        print("Processing", fp)
        human_df = pd.read_csv("../../results/greedy/all/non_verifiable/humans-2shot-filtered/normalized.csv", index_col=0)
        model_df = pd.read_csv(fp, index_col=0)
        dist = compute_proportional_agreement(model_df, human_df, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)
        dist["n_shots"] = n_shots
        
        model_name = fp.rpartition("shot/")[-1].rpartition("_normalized")[0]
        dist["model"] = model_name
        _non_verifiable_results_pa.append(dist)

_non_verifiable_results_pa = pd.concat(_non_verifiable_results_pa, axis=0).reset_index(drop=True)

Processing ../../results/greedy/all/non_verifiable/models-0shot/full__allenai__OLMo-7B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__google__gemma-1.1-2b-it_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__lmsys__vicuna-13b-v1.5_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__meta-llama__Meta-Llama-3-70B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__meta-llama__Meta-Llama-3-8B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__mistralai__Mistral-7B-Instruct-v0.2_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-4o-2024-05-13_normalized.csv
Processin

In [40]:
ind_pa_values = _non_verifiable_results_pa[
    (_non_verifiable_results_pa["n_shots"]==2) # change this w/ 0 to get the results for 0 shot
    & ~(_non_verifiable_results_pa["model"].isin(EXCLUDE_MODELS))
].pivot(index="uncertainty_expression", columns="model", values="distance")

ind_pa_values.loc["Avg PA", :] = ind_pa_values.mean(axis=0)
ind_pa_values.loc["Std PA", :] = ind_pa_values.std(axis=0)

print(table2latex(
    ind_pa_values,
    label="tab:app:pa-breakdown-expr:nv-2shot",
    caption="\\textbf{Proportional Agreement (PA) score per uncertainty expression in the non-verifiable setting}. The scores are with respect to the population-level human reference distribution (\\texttt{Human}+NV).",
    fmtter=format_as_percentage,
    normalize=(0,1)
))

\begin{table}
\caption{\textbf{Proportional Agreement (PA) score per uncertainty expression in the non-verifiable setting}. The scores are with respect to the population-level human reference distribution (\texttt{Human}+NV).}
\label{tab:tab:app:pa-breakdown-expr:nv-2shot}
\begin{tabular}{lllllllllll}
\toprule
model & full  allenai  OLMo-7B-Instruct & full  google  gemma-1.1-2b-it & full  meta-llama  Meta-Llama-3-70B-Instruct & full  meta-llama  Meta-Llama-3-8B-Instruct & gpt-3.5-turbo-0125 & gpt-4-turbo-2024-04-09 & gpt-4o-2024-05-13 & sampling  meta-llama  Llama-3-70b-chat-hf & sampling  mistralai  Mixtral-8x22B-Instruct-v0.1 & sampling  mistralai  Mixtral-8x7B-Instruct-v0.1 \\
uncertainty_expression &  &  &  &  &  &  &  &  &  &  \\
\midrule
almost certain & \cellcolor[HTML]{60ba6c}{55.0} & \cellcolor[HTML]{f6fcf4}{0.8} & \cellcolor[HTML]{4bb062}{60.0} & \cellcolor[HTML]{48ae60}{60.6} & \cellcolor[HTML]{5bb86a}{55.9} & \cellcolor[HTML]{48ae60}{60.6} & \cellcolor[HTML]{a8dca2}{35.5} &

/home/cbelem/projects/uncertainty-lm-lm-perception/llm-uncertainty-perceptions/notebooks/analysis/utils_latex.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  table_colored = table.applymap(lambda x: color_cell(x, cmap, norm))


In [41]:
_verifiable_results_pa = []

for n_shots in (0, 2):
    model_filepaths = sorted(glob.glob(f"../../results/greedy/all/verifiable/models-{n_shots}shot/*_normalized.csv"))  
    for fp in model_filepaths:
        print("Processing", fp)
        human_df = pd.read_csv("../../results/greedy/all/non_verifiable/humans-2shot-filtered/normalized.csv", index_col=0)
        model_df = pd.read_csv(fp, index_col=0)
        dist = compute_proportional_agreement(model_df, human_df, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)
        dist["n_shots"] = n_shots
        
        model_name = fp.rpartition("shot/")[-1].rpartition("_normalized")[0]
        dist["model"] = model_name
        _verifiable_results_pa.append(dist)

_verifiable_results_pa = pd.concat(_verifiable_results_pa, axis=0).reset_index(drop=True)
ind_pa_values = _verifiable_results_pa[
    (_verifiable_results_pa["n_shots"]==2) # change this w/ 0 to get the results for 0 shot
    & ~(_verifiable_results_pa["model"].isin(EXCLUDE_MODELS))
].pivot(index="uncertainty_expression", columns="model", values="distance")

ind_pa_values.loc["Avg PA", :] = ind_pa_values.mean(axis=0)
ind_pa_values.loc["Std PA", :] = ind_pa_values.std(axis=0)

print(table2latex(
    ind_pa_values,
    label="tab:app:pa-breakdown-expr:v-2shot",
    caption="\\textbf{Proportional Agreement (PA) score per uncertainty expression in the verifiable setting}. The scores are with respect to the population-level human reference distribution (\\texttt{Human}+NV).",
    fmtter=format_as_percentage,
    normalize=(0,1)
))

Processing ../../results/greedy/all/verifiable/models-0shot/full__allenai__OLMo-7B-Instruct_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/full__google__gemma-1.1-2b-it_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/full__lmsys__vicuna-13b-v1.5_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/full__meta-llama__Meta-Llama-3-8B-Instruct_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/full__mistralai__Mistral-7B-Instruct-v0.2_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/gpt-4o-2024-05-13_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/sampling__meta-llama__Llama-3-70b-chat-hf_normalized.csv
Processing ../../results/greedy/all/verifiable/

/home/cbelem/projects/uncertainty-lm-lm-perception/llm-uncertainty-perceptions/notebooks/analysis/utils_latex.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  table_colored = table.applymap(lambda x: color_cell(x, cmap, norm))


## 3. Mean Absolute Error

In [23]:
_non_verifiable_results_mae = []

for n_shots in (0, 2):
    model_filepaths = sorted(glob.glob(f"../../results/greedy/all/non_verifiable/models-{n_shots}shot/*_normalized.csv"))  
    for fp in model_filepaths:
        print("Processing", fp)
        human_df = pd.read_csv("../../results/greedy/all/non_verifiable/humans-2shot-filtered/normalized.csv", index_col=0)
        model_df = pd.read_csv(fp, index_col=0)
        dist = compute_mean_conf_error(model_df, human_df, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)
        dist["distance"] = dist["distance"].apply(np.abs)
        
        dist["n_shots"] = n_shots
        dist["setting"] = "non-verifiable"
        model_name = fp.rpartition("shot/")[-1].rpartition("_normalized")[0]
        dist["model"] = model_name
        _non_verifiable_results_mae.append(dist)

_non_verifiable_results_mae = pd.concat(_non_verifiable_results_mae, axis=0).reset_index(drop=True)

Processing ../../results/greedy/all/non_verifiable/models-0shot/full__allenai__OLMo-7B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__google__gemma-1.1-2b-it_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__lmsys__vicuna-13b-v1.5_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__meta-llama__Meta-Llama-3-70B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__meta-llama__Meta-Llama-3-8B-Instruct_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/full__mistralai__Mistral-7B-Instruct-v0.2_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-4o-2024-05-13_normalized.csv
Processin

In [45]:
from scipy.stats import wasserstein_distance

values = [-1] + [i for i in range(0, 101, 5)]
prob1 = [1] + [0 for i in range(0, 101, 5)]
prob2 = [0 for i in range(0, 101, 5)] + [1]

wasserstein_distance(u_values=values, v_values=values, u_weights=prob1, v_weights=prob2)

101.0